$$V_{LJ}(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^6\right]$$
$$\partial_rV_{LJ}(r) = 4\epsilon\left[\frac{-13}{\sigma} \left(\frac{\sigma}{r}\right)^{13}+\frac{7}{\sigma}\left(\frac{\sigma}{r}\right)^7\right]$$
$$\partial_rV_{LJ}(r_0) =0= 4\epsilon\left[\frac{-12}{\sigma} \left(\frac{\sigma}{r_0}\right)^{13}+\frac{6}{\sigma}\left(\frac{\sigma}{r_0}\right)^7\right]$$
$$\left(\frac{\sigma}{r_0}\right)^{6}=\frac{6}{12}$$
$$\sigma=\sqrt[6]{\frac{1}{2}}r_0\approx 0.891\cdot r_0 = 0.8908987181403393 \cdot r_0$$

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
from matplotlib.animation import FuncAnimation, PillowWriter 

def first_hot_then_cold(t):
    down_point = 1500
    up_point = 100
    width = 20
    return 5/(1+np.exp(1*(t-down_point)/width))/(1+np.exp(-1*(t-up_point)/width))+0.5
    # return 10 if t<100 else 0.9



def F_LJ(r,r0,epsilon=1): #Lennard Jones potential
    sigma = 0.5**(1/6)*r0
    s_over_r = sigma/r 
    return -4*epsilon*(7*s_over_r**7-13*s_over_r**13)/sigma

class Model():
    def __init__(self,temp_curve):
        # Parameters
        self.N = 20
        self.dt = 0.01
        self.T = 5000
        self.i = 1
        self.temp_curve=temp_curve
        # initial geometry of the system, in this case Triangular grid

        X,Y = np.meshgrid(range(self.N),range(int(self.N*2/np.sqrt(3))))

        X = np.reshape(X,X.size).astype(float)
        Y = np.reshape(Y,Y.size).astype(float)
        offsets = (Y.astype(int)%2)/2
        X += offsets
        Y*=np.sqrt(3)/2

        # plt.scatter(X,Y)
        # plt.show()
        P0 = np.array([X,Y])
        self.m = P0[1]*0+1

        self.P = np.zeros((self.T,P0.shape[0],P0.shape[1]))

        # initial conditions
        self.P[0] = P0
        self.P[1] = P0

 
    def contain(self):
        self.P[0:,:,][ self.P[0:,:,]<-1]=-1
        self.P[1:,:,][ self.P[1:,:,]<-1]=-1
        self.P[0:,:,][ self.P[0:,:,]>20]=20
        self.P[1:,:,][ self.P[1:,:,]>20]=20

    def distMat(self):
        self.dP = np.array([np.reshape(p,(len(p),1))-np.reshape(p,(1,len(p))) for p in self.P[self.i]])
        self.d  = np.sum(self.dP**2,axis=0)

    def force(self):
        self.d[np.diag_indices_from(self.d)]=1 #just to avoid divide by zero error message
        self.f = [dp/self.d*F_LJ(self.d,r0=1) for dp in self.dP]
        for F in self.f: # No self interaction
            # F +=np.random.normal(0,0.01,size=F.size)

            F[np.diag_indices_from(F)]=0

    def update(self,i):
        self.i = i
        self.distMat()
        self.force()
        self.contain()
        a = self.f@self.m 
        a+= np.random.normal(0,self.temp_curve(i),size=a.shape)
        self.P[i+1] = self.P[i]*2-self.P[i-1]+self.dt**2*a


system = Model(first_hot_then_cold)

def init(): 
    # fig.tight_layout()
    fig.set_size_inches((5,8))


    ax[0].set_xlim([-2,system.N+1])
    ax[0].set_ylim([-2,system.N+1])
    ax[0].set_aspect('equal')
    ax[0].set_axis_off()

    ax[1].set_xlim([0,system.T])
    ax[1].set_ylim([0,12])
    ax[1].set_xlabel("time")
    ax[1].set_ylabel("temperature")
    # ax[1].set_yscale('log')


def update(i):

    system.update(i)
    ax[0].set_title("$T="+str(round(system.temp_curve(i),1))+"$")

    L = system.P.shape[2]
    dots1[0].set_data(system.P[i,0,:int(L/2)+10],system.P[i,1,:int(L/2)+10])
    dots2[0].set_data(system.P[i,0,int(L/2)+10:],system.P[i,1,int(L/2)+10:])

    time_elapsed = list(range(1,i+1))
    tempPlot[0].set_data(time_elapsed,list(map(first_hot_then_cold,time_elapsed)))

system.update(1)

fig, ax = plt.subplots(2,1,  gridspec_kw={'height_ratios': [3, 1]})

dots1 =ax[0].plot(system.P[1][0],system.P[1][1], "o", color="blue")
dots2 =ax[0].plot(system.P[1][0],system.P[1][1], "o", color="red")
tempPlot = ax[1].plot([],[])

time = range(1,int(system.T-1))
ani = FuncAnimation(fig, update, time, init_func=init)  


writer = PillowWriter(fps=25)
ani.save("heelmooi3.gif", writer=writer)

